In [ ]:
import scrapy
import pandas as pd
from scrapy.crawler import CrawlerProcess
from scrapy import Request
from scrapy.spiders import Spider
from datetime import datetime

In [ ]:
import os
from mongodb_client import MongoDBClient
mongo_uri = os.getenv("MONGO_URI")
db_name = os.getenv("DB_NAME")
collection = os.getenv("COLLECTION_NAME")

In [ ]:

client = MongoDBClient(mongo_uri, db_name, collection)

In [ ]:
class ElDiarioSpider(Spider):
    name = "eldiario"
    allowed_domains = ["eldiario.net"]
    start_urls = [
        f"https://www.eldiario.net/portal/page/{i}/?s=feminicidio" for i in range(1, 62)
        
    ]
    def __init__(self):
        self.items = []
        self.mongo_client = client
    def date_formatter(self, date_str, date_format="%Y-%m-%d"):
        try:
            new_date = datetime.strptime(date_str, date_format)
            return new_date
        except Exception as e:
            self.logger.error(f"Error al formatear fecha: {e}")
            return None

    def tittle_formatter(self, title):
        try:
            new_title = title.replace("“", '"').replace("”", '"')
            return new_title
        except Exception as e:
            self.logger.error(f"Error al formatear título: {e}")
            return title

    def tag_formatter(self, tags):
        try:
            list_tags = [t.lower().strip() for t in tags.split("-")]
            return list_tags
        except Exception as e:
            self.logger.error(f"Error al formatear tags: {e}")
            return tags
    
    def section_formatter(self, tags):
        try:
            if len(tags) > 1:
                section = tags[1]
            else:
                section = tags[0]
            return section
        except Exception as e:
            self.logger.error(f"Error al formatear sección: {e}")
            return tags

    def body_formatter(self, body):
        try:
            new_body = [b for b in body if "Lea tamb" not in b and b.strip()]
            new_body = [
                b.strip()
                .replace("\xa0", " ")
                .replace("\ufeff", " ")
                .replace("“", '"')
                .replace("”", '"')
                .replace("\u200b", " ")
                for b in new_body
            ]
            new_body = [b for b in new_body if b != " "]
            return new_body
        except Exception as e:
            self.logger.error(f"Error al formatear cuerpo: {e}")
            return body

    def start_requests(self):
        for url in self.start_urls:
            self.logger.info(f"Enviando request a: {url}")
            yield Request(url=url, callback=self.parse_response)

    def parse_response(self, response):
        self.logger.info(f"Recibida respuesta: {response.url}")
        try:
            noticias = response.xpath("//h3[contains(@class, 'entry-title')]/a/@href").getall()
            self.logger.info(f"Total noticias encontradas: {len(noticias)}")
            for noticia in noticias:
                self.logger.info(f"Enviando request a: {noticia}")
                yield Request(url=noticia, callback=self.parse_news)
        except Exception as e:
            self.logger.error(f"Error al procesar la respuesta JSON: {e}")
            return

    def parse_news(self, response):
        title = response.xpath("//h1[@class='tdb-title-text']/text()").get()
        item = {}
        item["url"] = response.url
        item["title"] = self.tittle_formatter(title)
        section = response.xpath("//a[@class='tdb-entry-category']/text()").get()
        item["tags"] = self.tag_formatter(section)
        item["section"] = self.section_formatter(item["tags"])
        body = [
            p.xpath("string(.)").get()
            for p in response.xpath("//div[contains(@class, 'td-fix-index')]/p")
        ]
        item["body"] = self.body_formatter(body)
        item["tags"] = self.tag_formatter(section)
        
        date_published = response.xpath("//time[contains(@class, 'td-module-date')]/text()").get()
        item["date_published"] = self.date_formatter(date_published,"%d/%m/%Y")
        item["source"] = "eldiario"
        if "seguridad" in  item["tags"]:
            self.items.append(item)
            self.logger.info(f"Noticia agregada: {item['title']}")

    def close(self, reason):
        self.mongo_client.connect()
        self.logger.info("Guardando datos en MongoDB")
        for item in self.items:
            try:
                self.mongo_client.insert_new_document(item, "url")
            except Exception as e:
                self.logger.error(f"Error al insertar en MongoDB: {e}")
        self.mongo_client.close()
        self.logger.info(f"Spider cerrado por la razón: {reason}")

In [ ]:
process = CrawlerProcess()
process.crawl(ElDiarioSpider)
process.start()